# SafeVision Notebook 7: Fine-tuning Florence-2 to describe construction scenes (experiment E11)

**Goal:** the box detector says *who* is missing a helmet, gloves or glasses, but not *what is going on* (working at height, next to an excavator, cutting steel...). That context decides which accident is likely. We teach a small vision-language model, **Florence-2-base** (0.23 B parameters, Microsoft, MIT licence) [F1], to write construction-safety captions, using the **human-written captions of ConstructionSite 10k** [1].

**Why fine-tune and not just use it as it is?** Pretrained Florence-2 writes general captions ("*a man standing next to a building*"). The ConstructionSite captions were written by experts and mention PPE, machines and hazards. Fine-tuning moves the model towards that style. We measure the difference.

| ID | Question | Main scores |
|---|---|---|
| **E11a** | Does fine-tuning make captions closer to expert captions? | BLEU-4, ROUGE-L, CIDEr, BERTScore-F1 (pretrained vs fine-tuned, same test images) |
| **E11b** | Does it mention the right safety items (helmet, vest, gloves, harness, glasses...)? | PPE-term precision, recall, F1 + invented PPE terms |
| **E11c** | Is it fast enough for the app? | ms per image |

**Training setup** (follows common Florence-2 fine-tuning practice [F2]): the image encoder (DaViT) is **frozen**, the language part is trained with the normal next-word loss, AdamW, mixed precision. The **test** split is never used for training. A small part of the train split (5 %) is kept for choosing the best epoch.

**Runtime:** T4 GPU, about 1-1.5 h. The model is saved to the HF repo after every epoch, so a disconnected session resumes from the last saved epoch.

## 0. Install libraries
Florence-2 is built into `transformers` from version 4.56 (no `trust_remote_code` needed). `pycocoevalcap` gives the standard captioning scores (BLEU, ROUGE-L, CIDEr) [F3], `bert-score` the meaning-based score.

In [ ]:
# [colab-only]
!nvidia-smi
%pip install -q -U "transformers>=4.56" datasets accelerate pycocoevalcap bert-score openpyxl huggingface_hub

## 1. Configuration (edit this cell only)

In [ ]:
# CONFIG
from pathlib import Path
import re, json, random, time, os, importlib.util

ON_KAGGLE = Path("/kaggle/input").exists()
IN_COLAB = importlib.util.find_spec("google.colab") is not None
BASE = Path("/kaggle/working") if ON_KAGGLE else Path("/content") if IN_COLAB else Path.home() / "safevision"
OUT = BASE / "safevision_outputs_nb7"
CS10K_HUB = "LouisChen15/ConstructionSite"
CS10K_LOCAL = None                     # path of a dataset saved with save_to_disk (optional)

MODEL_ID = "florence-community/Florence-2-base"    # native transformers version of microsoft/Florence-2-base
TASK = "<MORE_DETAILED_CAPTION>"                   # Florence-2 task token used for training and testing
N_TRAIN = None          # None = whole train split (about 7,000 photos); a number = random subset (faster)
VAL_FRAC = 0.05         # part of the train split kept for choosing the best epoch
N_TEST = 500            # test photos scored in E11 (same photos for both models)
EPOCHS = 3; BATCH = 8; GRAD_ACC = 2; LR = 1e-5; WARMUP = 0.05; WEIGHT_DECAY = 0.01
FREEZE_VISION = True    # keep the image encoder fixed (faster, less memory, less forgetting)
MAX_CAPTION_TOKENS = 160
GEN_BEAMS = 3; GEN_MAX_NEW = 160
RUN_BERTSCORE = True
NUM_WORKERS = 2
SEED = 42

RES, FIGS, MODELS = OUT / "results", OUT / "figures", OUT / "models"
CKPT = BASE / "florence2_ckpt"           # local copy of the last finished epoch
for p in [OUT, RES, FIGS, MODELS, CKPT]:
    p.mkdir(parents=True, exist_ok=True)
random.seed(SEED)
HF_REPO, token = None, None

def get_secret(name):
    # Environment variable, Colab Secrets or Kaggle Secrets (None if not set). Keys are never printed.
    if os.environ.get(name):
        return os.environ[name]
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        pass
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return None

## 2. Results logger

In [ ]:
# Results logger
import csv, datetime
import pandas as pd
LOG_CSV = RES / "experiments_log.csv"
LOG_FIELDS = ["experiment", "run", "model", "dataset", "split", "metric", "value", "notes", "timestamp"]

def log_result(experiment, run, model, metric, value, notes="", dataset="ConstructionSite10k", split="test"):
    new_file = not LOG_CSV.exists()
    with open(LOG_CSV, "a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=LOG_FIELDS)
        if new_file:
            w.writeheader()
        w.writerow(dict(experiment=experiment, run=run, model=model, dataset=dataset, split=split, metric=metric,
                        value=round(float(value), 4), notes=notes, timestamp=datetime.datetime.now().isoformat(timespec="seconds")))

def read_log():
    return pd.read_csv(LOG_CSV) if LOG_CSV.exists() else pd.DataFrame(columns=LOG_FIELDS)

## 3. Log in to Hugging Face
Needed for the gated ConstructionSite 10k dataset and for saving checkpoints to the private repo `safevision-artifacts`.

In [ ]:
# [colab-only]
from huggingface_hub import login, HfApi
token = get_secret("HF_TOKEN")
login(token=token)
HF_REPO = f"{HfApi(token=token).whoami()['name']}/safevision-artifacts"
print("HF repo:", HF_REPO)

## 4. Load ConstructionSite 10k and pick the photos
Train split → training + 5 % validation. Test split → `N_TEST` random photos for E11 (fixed seed). Photos without a caption are skipped.

In [ ]:
# Load ConstructionSite 10k and pick the photos
from datasets import load_dataset, load_from_disk
ds = load_from_disk(CS10K_LOCAL) if CS10K_LOCAL else load_dataset(CS10K_HUB, token=token)
print(ds)

def has_caption(c):
    return isinstance(c, str) and len(c.split()) >= 5

rng = random.Random(SEED)
train_ids = [i for i, c in enumerate(ds["train"]["image_caption"]) if has_caption(c)]
rng.shuffle(train_ids)
if N_TRAIN:
    train_ids = train_ids[:N_TRAIN]
n_val = max(20, int(len(train_ids) * VAL_FRAC))
val_ids, train_ids = train_ids[:n_val], train_ids[n_val:]
test_ids = [i for i, c in enumerate(ds["test"]["image_caption"]) if has_caption(c)]
rng.shuffle(test_ids); test_ids = sorted(test_ids[:N_TEST])
lens = pd.Series([len(ds["train"][i]["image_caption"].split()) for i in train_ids[:300]])
print(f"train {len(train_ids)} | val {len(val_ids)} | test {len(test_ids)} | caption words: median {lens.median():.0f}, max {lens.max()}")
print("Example caption:", ds["train"][train_ids[0]]["image_caption"][:400])

## 5. Load Florence-2-base

In [ ]:
# Load Florence-2-base
import torch
from transformers import AutoProcessor, Florence2ForConditionalGeneration
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
processor = AutoProcessor.from_pretrained(MODEL_ID)

def load_model(path):
    # float32 weights: the checkpoint is stored in float16, and mixed-precision training needs float32 master weights
    m = Florence2ForConditionalGeneration.from_pretrained(path, dtype=torch.float32).to(DEVICE)
    return m

def n_params(m, trainable=False):
    return sum(p.numel() for p in m.parameters() if p.requires_grad or not trainable)

## 6. Caption helper and scores (used for both models)
- **BLEU-4 / ROUGE-L / CIDEr**: word-overlap scores used in image-captioning research [F3]; CIDEr is the main one.
- **BERTScore-F1**: meaning similarity, so a correct caption with other words is not punished [F4].
- **PPE terms (E11b)**: for each photo we check which safety items the expert caption mentions and which the model mentions. *Recall* = items the expert mentioned that the model also mentioned; *precision* = items the model mentioned that the expert also mentioned; *invented* = share of photos where the model names a safety item the expert did not.

In [ ]:
# Caption helper and scores (used for both models)
import numpy as np
from torch.utils.data import Dataset, DataLoader

def caption_images(model, images, batch=8):
    # Captions for a list of PIL images (beam search).
    model.eval(); out = []
    dtype = torch.float16 if DEVICE == "cuda" else torch.float32
    for i in range(0, len(images), batch):
        imgs = [im.convert("RGB") for im in images[i:i + batch]]
        inp = processor(text=[TASK] * len(imgs), images=imgs, return_tensors="pt", padding=True).to(DEVICE)
        with torch.no_grad(), torch.autocast(DEVICE, dtype=dtype, enabled=DEVICE == "cuda"):
            ids = model.generate(input_ids=inp["input_ids"], pixel_values=inp["pixel_values"],
                                 max_new_tokens=GEN_MAX_NEW, num_beams=GEN_BEAMS)
        out += [t.strip() for t in processor.batch_decode(ids, skip_special_tokens=True)]
    return out

PPE_TERMS = {
    "helmet": r"hard ?hats?|hardhats?|helmets?|safety hats?|head protection",
    "vest": r"vests?|high[- ]vis\w*|hi[- ]vis|reflective (?:vest|clothing|jacket)s?",
    "gloves": r"gloves?",
    "harness": r"harness\w*|lanyards?|fall (?:arrest|protection)",
    "glasses": r"goggles?|safety glasses|eye protection|face shields?",
    "mask": r"masks?|respirators?",
    "boots": r"boots?|safety shoes",
}
PPE_RE = {k: re.compile(rf"\b({v})\b", re.I) for k, v in PPE_TERMS.items()}

def ppe_set(text):
    return {k for k, r in PPE_RE.items() if r.search(text or "")}

def ppe_scores(refs, hyps):
    tp = fp = fn = invented = 0
    for r, h in zip(refs, hyps):
        R, H = ppe_set(r), ppe_set(h)
        tp += len(R & H); fp += len(H - R); fn += len(R - H); invented += bool(H - R)
    p = tp / (tp + fp) if tp + fp else 0.0; rc = tp / (tp + fn) if tp + fn else 0.0
    return {"ppe_precision": p, "ppe_recall": rc, "ppe_F1": 2 * p * rc / (p + rc) if p + rc else 0.0,
            "ppe_invented_pct": 100 * invented / max(1, len(refs))}

def norm(t):
    return " ".join(re.findall(r"[a-z0-9]+", (t or "").lower()))

def caption_scores(refs, hyps):
    from pycocoevalcap.bleu.bleu import Bleu
    from pycocoevalcap.rouge.rouge import Rouge
    from pycocoevalcap.cider.cider import Cider
    gts = {i: [norm(r)] for i, r in enumerate(refs)}; res = {i: [norm(h) or "empty"] for i, h in enumerate(hyps)}
    s = {"BLEU4": Bleu(4).compute_score(gts, res, verbose=0)[0][3], "ROUGE_L": Rouge().compute_score(gts, res)[0],
         "CIDEr": Cider().compute_score(gts, res)[0]}
    if RUN_BERTSCORE:
        from bert_score import score as bscore
        s["BERTScore_F1"] = float(bscore(hyps, refs, lang="en", rescale_with_baseline=True, verbose=False)[2].mean())
    s.update(ppe_scores(refs, hyps))
    s["words_per_caption"] = float(np.mean([len(h.split()) for h in hyps]))
    return s

## 7. Experiment E11 (before): Pretrained Florence-2 on the test photos

In [ ]:
# Experiment E11 (before): Pretrained Florence-2 on the test photos
test_imgs = [ds["test"][i]["image"] for i in test_ids]
test_refs = [ds["test"][i]["image_caption"] for i in test_ids]
pre_file = RES / "E11_captions_pretrained.json"
if pre_file.exists():
    pre_caps = json.load(open(pre_file)); pre_ms = json.load(open(RES / "E11_pretrained_ms.json"))
else:
    base_model = load_model(MODEL_ID)
    t0 = time.time(); pre_caps = caption_images(base_model, test_imgs); pre_ms = 1000 * (time.time() - t0) / len(test_imgs)
    json.dump(pre_caps, open(pre_file, "w")); json.dump(pre_ms, open(RES / "E11_pretrained_ms.json", "w"))
    del base_model; torch.cuda.empty_cache()
pre_scores = caption_scores(test_refs, pre_caps)
print(json.dumps({k: round(v, 3) for k, v in pre_scores.items()}, indent=1))

## 8. Fine-tuning
Each training example: photo + task token → expert caption. Loss = standard next-word cross-entropy on the caption. After every epoch: validation loss, save the model to HF `checkpoints/florence2_cs10k/` (resume point) and keep the best epoch.

In [ ]:
# Fine-tuning
from transformers import get_cosine_schedule_with_warmup
from huggingface_hub import snapshot_download

class CaptionSet(Dataset):
    def __init__(self, split, ids):
        self.split, self.ids = split, ids
    def __len__(self):
        return len(self.ids)
    def __getitem__(self, i):
        r = ds[self.split][self.ids[i]]
        return r["image"].convert("RGB"), r["image_caption"]

def collate(batch):
    imgs, caps = zip(*batch)
    inp = processor(text=[TASK] * len(imgs), images=list(imgs), return_tensors="pt", padding=True)
    lab = processor.tokenizer(list(caps), return_tensors="pt", padding=True, truncation=True, max_length=MAX_CAPTION_TOKENS).input_ids
    lab[lab == processor.tokenizer.pad_token_id] = -100            # padding is not scored
    return inp["input_ids"], inp["pixel_values"], lab

train_dl = DataLoader(CaptionSet("train", train_ids), batch_size=BATCH, shuffle=True, collate_fn=collate, num_workers=NUM_WORKERS)
val_dl = DataLoader(CaptionSet("train", val_ids), batch_size=BATCH, shuffle=False, collate_fn=collate, num_workers=NUM_WORKERS)

# ---- resume: last finished epoch from HF (or local) ----
state = {"epoch": 0, "best_val": 1e9, "history": []}
start_path = MODEL_ID
if HF_REPO:
    try:
        p = Path(snapshot_download(HF_REPO, token=token, allow_patterns=["checkpoints/florence2_cs10k/**"])) / "checkpoints/florence2_cs10k"
        if (p / "state.json").exists():
            state = json.load(open(p / "state.json")); start_path = str(p / "last")
            print("Resuming after epoch", state["epoch"])
    except Exception as e:
        print("No checkpoint on HF (normal on the first run):", str(e)[:80])
if (CKPT / "state.json").exists() and json.load(open(CKPT / "state.json"))["epoch"] > state["epoch"]:
    state = json.load(open(CKPT / "state.json")); start_path = str(CKPT / "last")

model = load_model(start_path)
if FREEZE_VISION:
    for n, p in model.named_parameters():
        if "vision_tower" in n:
            p.requires_grad = False
print(f"parameters: {n_params(model) / 1e6:.0f} M, trainable: {n_params(model, True) / 1e6:.0f} M")
opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR, weight_decay=WEIGHT_DECAY)
total = EPOCHS * len(train_dl) // GRAD_ACC
sched = get_cosine_schedule_with_warmup(opt, int(WARMUP * total), total)
for _ in range(state["epoch"] * len(train_dl) // GRAD_ACC):        # put the learning-rate schedule where it stopped
    sched.step()
scaler = torch.amp.GradScaler(enabled=DEVICE == "cuda")

def val_loss():
    model.eval(); tot = n = 0
    with torch.no_grad():
        for ids, px, lab in val_dl:
            with torch.autocast(DEVICE, dtype=torch.float16, enabled=DEVICE == "cuda"):
                loss = model(input_ids=ids.to(DEVICE), pixel_values=px.to(DEVICE), labels=lab.to(DEVICE)).loss
            tot += loss.item() * len(ids); n += len(ids)
    return tot / max(1, n)

def save_epoch(is_best):
    model.save_pretrained(CKPT / "last"); processor.save_pretrained(CKPT / "last")
    if is_best:
        model.save_pretrained(MODELS / "florence2_captioner"); processor.save_pretrained(MODELS / "florence2_captioner")
    json.dump(state, open(CKPT / "state.json", "w"))
    if HF_REPO:
        try:
            api = HfApi(token=token)
            api.upload_folder(folder_path=str(CKPT), repo_id=HF_REPO, path_in_repo="checkpoints/florence2_cs10k")
            if is_best:
                api.upload_folder(folder_path=str(MODELS / "florence2_captioner"), repo_id=HF_REPO,
                                  path_in_repo="checkpoints/florence2_cs10k/best")
        except Exception as e:
            print("checkpoint upload failed:", e)

for epoch in range(state["epoch"] + 1, EPOCHS + 1):
    model.train(); t0 = time.time(); run = 0.0
    for step, (ids, px, lab) in enumerate(train_dl, 1):
        with torch.autocast(DEVICE, dtype=torch.float16, enabled=DEVICE == "cuda"):
            loss = model(input_ids=ids.to(DEVICE), pixel_values=px.to(DEVICE), labels=lab.to(DEVICE)).loss / GRAD_ACC
        scaler.scale(loss).backward(); run += loss.item() * GRAD_ACC
        if step % GRAD_ACC == 0 or step == len(train_dl):
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); opt.zero_grad(); sched.step()
        if step % 100 == 0:
            print(f"epoch {epoch} step {step}/{len(train_dl)} loss {run / step:.3f} ({time.time() - t0:.0f} s)")
    vl = val_loss(); is_best = vl < state["best_val"]
    state["history"].append({"epoch": epoch, "train_loss": run / len(train_dl), "val_loss": vl, "minutes": (time.time() - t0) / 60})
    state["epoch"] = epoch; state["best_val"] = min(vl, state["best_val"])
    if is_best:
        state["best_epoch"] = epoch
    print(f"epoch {epoch}: train {run / len(train_dl):.3f} | val {vl:.3f} {'(best)' if is_best else ''}")
    save_epoch(is_best)

hist = pd.DataFrame(state["history"]); hist.to_csv(RES / "E11_training_history.csv", index=False); print(hist)

## 9. Experiment E11 (after): Fine-tuned Florence-2 on the same test photos

In [ ]:
# Experiment E11 (after): Fine-tuned Florence-2 on the same test photos
best_dir = MODELS / "florence2_captioner"
if not (best_dir / "config.json").exists() and HF_REPO:          # e.g. the best epoch was saved in an earlier session
    p = Path(snapshot_download(HF_REPO, token=token, allow_patterns=["checkpoints/florence2_cs10k/best/**"]))
    import shutil; shutil.copytree(p / "checkpoints/florence2_cs10k/best", best_dir, dirs_exist_ok=True)
del model; torch.cuda.empty_cache()
ft_model = load_model(best_dir)
t0 = time.time(); ft_caps = caption_images(ft_model, test_imgs); ft_ms = 1000 * (time.time() - t0) / len(test_imgs)
json.dump(ft_caps, open(RES / "E11_captions_finetuned.json", "w"))
ft_scores = caption_scores(test_refs, ft_caps)

table = pd.DataFrame({"pretrained": pre_scores, "fine-tuned": ft_scores}).round(4)
table.loc["ms_per_image"] = [round(pre_ms, 1), round(ft_ms, 1)]
for model_name, col in [("Florence-2-base (pretrained)", "pretrained"), ("Florence-2-base (fine-tuned CS10k)", "fine-tuned")]:
    for metric, v in table[col].items():
        exp = "E11c" if metric == "ms_per_image" else "E11b" if metric.startswith("ppe") else "E11a"
        log_result(exp, f"{exp}_{col}", model_name, metric, v, f"{len(test_ids)} test photos, {GEN_BEAMS} beams")
table.to_csv(RES / "E11_caption_scores.csv"); print(table)

### Side-by-side examples and charts

In [ ]:
# Side-by-side examples and charts
import matplotlib.pyplot as plt, textwrap
fig, axes = plt.subplots(3, 1, figsize=(12, 15))
for ax, k in zip(axes, [0, len(test_ids) // 2, len(test_ids) - 1]):
    ax.imshow(test_imgs[k]); ax.axis("off")
    ax.set_title("\n".join(["EXPERT: " + textwrap.shorten(test_refs[k], 170), "PRETRAINED: " + textwrap.shorten(pre_caps[k], 170),
                            "FINE-TUNED: " + textwrap.shorten(ft_caps[k], 170)]), fontsize=8, loc="left")
plt.tight_layout(); plt.savefig(FIGS / "E11_examples.png", dpi=120); plt.show()

show = [m for m in ["BLEU4", "ROUGE_L", "CIDEr", "BERTScore_F1", "ppe_F1"] if m in table.index]
ax = table.loc[show, ["pretrained", "fine-tuned"]].plot.bar(figsize=(8, 4), rot=0, color=["#94a3b8", "#2563eb"])
ax.set_title(f"E11: Caption quality on {len(test_ids)} ConstructionSite test photos"); ax.grid(axis="y", alpha=.3)
plt.tight_layout(); plt.savefig(FIGS / "E11_scores.png", dpi=150); plt.show()
pd.DataFrame({"expert": test_refs, "pretrained": pre_caps, "fine_tuned": ft_caps}).to_csv(RES / "E11_all_captions.csv", index=False)

## 10. Save everything
The fine-tuned model goes to HF `nb7_florence2_captioner/models/florence2_captioner`. For the app, copy that folder to `backend/models/florence2_captioner`: the backend then adds the scene caption to the situation text automatically.

In [ ]:
# Save everything
import shutil
json.dump({"model_id": MODEL_ID, "task": TASK, "best_epoch": state.get("best_epoch"), "epochs": EPOCHS, "lr": LR,
           "freeze_vision": FREEZE_VISION, "n_train": len(train_ids), "n_test": len(test_ids)},
          open(RES / "E11_config.json", "w"), indent=1)
with pd.ExcelWriter(RES / "E11_florence2_captions.xlsx") as xw:
    read_log().to_excel(xw, "all_results", index=False)
    table.reset_index().rename(columns={"index": "metric"}).to_excel(xw, "E11_scores", index=False)
    hist.to_excel(xw, "training_history", index=False)
archive = shutil.make_archive(str(OUT.parent / "safevision_nb7_outputs"), "zip", OUT, ".")
print("Saved", archive)

In [ ]:
# [colab-only]
if HF_REPO:
    HfApi(token=token).upload_folder(folder_path=str(OUT), repo_id=HF_REPO, path_in_repo="nb7_florence2_captioner")
    print("Uploaded to", HF_REPO, "/ nb7_florence2_captioner")

## References
- [1] X. Chen et al., "ConstructionSite 10k" dataset (captions, rule violations, condition tags), 2025.
- [F1] B. Xiao et al., "Florence-2: Advancing a unified representation for a variety of vision tasks," *CVPR*, 2024.
- [F2] Hugging Face / Roboflow guides on fine-tuning Florence-2 (frozen vision encoder, small learning rate), 2024.
- [F3] R. Vedantam, C. L. Zitnick and D. Parikh, "CIDEr: Consensus-based image description evaluation," *CVPR*, 2015.
- [F4] T. Zhang et al., "BERTScore: Evaluating text generation with BERT," *ICLR*, 2020.